# Tema 3 · Localizar y delimitar piezas

Seis actividades sincronizadas con el PDF. Ejecuta en orden; guarda una copia. CPU, sin descargas y con las bibliotecas del curso.

Entrenaremos una regresión logística para clasificar cada píxel como pieza o fondo. Después, componentes conexas producirán regiones y cajas. No es un detector profundo ni un segmentador de instancias: aprenderemos también por qué falla con piezas unidas. Las arquitecturas profundas se explican en el PDF.

Todas las imágenes y anotaciones se generan por código. No son fotografías ni demuestran utilidad industrial. Solo existe una clase de interés: **pieza**; las formas no son clases predichas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects

# Escenas independientes por semilla. Las coordenadas son filas y columnas.
def escena(seed, contacto=False, bajo_contraste=False):
    rng = np.random.default_rng(seed)
    h, w = 128, 192
    yy, xx = np.mgrid[:h, :w]
    instancias = np.zeros((h, w), dtype=np.int32)
    centros = [(35, 37), (84, 95), (37, 153)]
    if contacto:
        centros = [(60, 65), (60, 94), (60, 123)]
    for k, (cy, cx) in enumerate(centros, start=1):
        if not contacto:
            cy, cx = cy + rng.integers(-6, 7), cx + rng.integers(-6, 7)
        radio = 20 if contacto else rng.uniform(15, 21)
        x, y = xx - cx, yy - cy
        if k == 1:
            dist = np.sqrt(x*x + y*y)
            mascara = (dist < radio) & (dist > radio * 0.45)
        elif k == 2:
            mascara = (np.abs(x) < radio) & (np.abs(y) < radio * 0.7)
        else:
            mascara = ((np.abs(x) < radio) & (np.abs(y) < radio)
                       & ((x < -radio * 0.2) | (y > radio * 0.2)))
        instancias[mascara] = k  # Si se solapan, la última es visible delante.
    fondo = np.array([0.35, 0.40, 0.44])
    color = np.array([0.44, 0.48, 0.52]) if bajo_contraste else np.array([0.25, 0.57, 0.74])
    rgb = np.where((instancias > 0)[..., None], color, fondo)
    rgb = rgb * rng.uniform(0.85, 1.15)
    rgb = np.clip(rgb + rng.normal(0, 0.065, rgb.shape), 0, 1).astype(np.float32)
    return {"id": seed, "rgb": rgb, "instancias": instancias,
            "mascara": instancias > 0}

def cajas_de_etiquetas(etiquetas):
    # regionprops devuelve (fila_min, col_min, fila_max, col_max).
    # Convención de caja: (xmin, ymin, xmax, ymax), extremos finales excluidos.
    return np.array([(p.bbox[1], p.bbox[0], p.bbox[3], p.bbox[2])
                     for p in regionprops(etiquetas)], dtype=float).reshape(-1, 4)

def mostrar_cajas(ax, rgb, cajas, color="lime"):
    ax.imshow(rgb)
    for xmin, ymin, xmax, ymax in cajas:
        ax.add_patch(Rectangle((xmin - 0.5, ymin - 0.5), xmax-xmin, ymax-ymin,
                               fill=False, edgecolor=color, linewidth=2))
    ax.axis("off")

def iou_mascara(a, b):
    union = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / union if union else 1.0

## Actividad 1 · Etiqueta, caja y máscara (10 min)

Compara una escena con sus anotaciones. La referencia se conoce porque construimos el dibujo, no porque el modelo la haya descubierto. La máscara semántica reúne todas las piezas; el mapa de instancias da un identificador a cada pieza visible.

In [ ]:
ejemplo = escena(50)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
mostrar_cajas(axes[0], ejemplo["rgb"], cajas_de_etiquetas(ejemplo["instancias"]))
axes[0].set_title("Cajas de referencia")
axes[1].imshow(ejemplo["mascara"], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Semántica: pieza / fondo")
axes[2].imshow(ejemplo["instancias"], cmap="tab10", vmin=0, vmax=9)
axes[2].set_title("Instancias: identificadores")
axes[3].imshow(ejemplo["rgb"])
axes[3].set_title("Imagen · contiene piezas")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Cajas (xmin, ymin, xmax, ymax):", cajas_de_etiquetas(ejemplo["instancias"]))

**Explica:** ¿qué anotación necesitarías para contar piezas? ¿Y para estimar el área visible de material de una arandela, sin contar su agujero?

**Tu respuesta:** …

## Actividad 2 · Aprender pieza/fondo (12 min)

Usamos escenas de entrenamiento con semillas 100–111, validación 200–203 y prueba 300–305. No comparten imágenes. Entrenamos con 1500 píxeles muestreados al azar por escena: solo usamos RGB, sin coordenadas ni información de vecinos. El ejemplo anterior no forma parte de la evaluación.

In [ ]:
entrenamiento = [escena(i) for i in range(100, 112)]
validacion = [escena(i) for i in range(200, 204)]
prueba = [escena(i) for i in range(300, 306)]
assert not ({e["id"] for e in entrenamiento} & {e["id"] for e in validacion + prueba})
rng = np.random.default_rng(42)
X, y = [], []
for e in entrenamiento:
    ids = rng.choice(e["mascara"].size, size=1500, replace=False)
    X.append(e["rgb"].reshape(-1, 3)[ids])
    y.append(e["mascara"].ravel()[ids])
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300))
modelo.fit(np.concatenate(X), np.concatenate(y))
def probabilidad(e):
    return modelo.predict_proba(e["rgb"].reshape(-1, 3))[:, 1].reshape(e["mascara"].shape)
prob = probabilidad(validacion[0])
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(validacion[0]["rgb"])
axes[0].set_title("Escena de validación")
im = axes[1].imshow(prob, vmin=0, vmax=1, cmap="viridis")
axes[1].set_title("Puntuación de pieza por píxel")
fig.colorbar(im, ax=axes[1])
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

**Explica:** ¿por qué este modelo puede fallar si pieza y fondo tienen colores parecidos? La salida entre 0 y 1 es una estimación del modelo; no garantiza probabilidad calibrada en nuevas condiciones.

**Tu respuesta:** …

## Actividad 3 · Umbral y limpieza (13 min)

Un umbral transforma la puntuación en máscara. Comparamos umbrales solo en validación mediante IoU de la clase pieza. Eliminamos regiones menores de 30 píxeles; este tamaño se fija antes de prueba. No rellenamos agujeros: una arandela tiene uno real.

In [ ]:
AREA_MINIMA = 30
UMBRALES = [0.25, 0.5, 0.75]
def segmentar(e, umbral, area_minima=AREA_MINIMA):
    mascara = probabilidad(e) >= umbral
    return remove_small_objects(mascara, min_size=area_minima, connectivity=1)
resultados_val = []
for umbral in UMBRALES:
    valor = np.mean([iou_mascara(e["mascara"], segmentar(e, umbral)) for e in validacion])
    resultados_val.append(valor)
    print("Umbral", umbral, "IoU media de validación:", round(valor, 3))
umbral_elegido = UMBRALES[int(np.argmax(resultados_val))]
print("Umbral elegido:", umbral_elegido)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, umbral in zip(axes, UMBRALES):
    ax.imshow(segmentar(validacion[0], umbral), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Umbral {umbral}")
    ax.axis("off")
plt.tight_layout()
plt.show()

**Modifica:** compara el resultado sin limpieza (`area_minima=0`) en validación. ¿Qué pasa con regiones pequeñas? ¿Y si un defecto que nos interesa fuese menor que 30 píxeles? No cambies la regla tras mirar prueba.

**Tu respuesta:** …

## Actividad 4 · De regiones a cajas (12 min)

Una componente conexa agrupa píxeles vecinos de primer orden (arriba, abajo, izquierda y derecha). Es una agrupación geométrica; no sabe qué es una pieza. Comparamos máscaras y cajas usando intersección sobre unión (IoU).

In [ ]:
def iou_caja(a, b):
    ancho = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    alto = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ancho * alto
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - inter
    return inter / union if union else 0.0

def detectar(e):
    mascara = segmentar(e, umbral_elegido)
    return mascara, cajas_de_etiquetas(label(mascara, connectivity=1))

mascara, cajas = detectar(validacion[0])
fig, ax = plt.subplots(figsize=(8, 4))
mostrar_cajas(ax, validacion[0]["rgb"], cajas)
ax.set_title(f"Cajas obtenidas: {len(cajas)}")
plt.show()
a = np.array([0, 0, 10, 10])
b = np.array([5, 0, 15, 10])
print("IoU de dos cajas de ejemplo:", iou_caja(a, b))
assert np.isclose(iou_caja(a, a), 1)
assert np.isclose(iou_caja(a, b), 1/3)

**Comprueba:** las cajas del ejemplo tienen 50 píxeles de intersección y 150 de unión. ¿Por qué el área de una caja no equivale al área visible de una arandela?

**Tu respuesta:** …

## Actividad 5 · Una máscara no separa instancias (10 min)

Creamos una escena de diagnóstico con piezas en contacto. El número de componentes puede ser inferior al de piezas visibles aunque la máscara pieza/fondo sea buena. Las referencias son regiones visibles; no estimamos la parte oculta.

In [ ]:
contacto = escena(400, contacto=True)
mascara_contacto, cajas_contacto = detectar(contacto)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(contacto["instancias"], cmap="tab10", vmin=0, vmax=9)
axes[0].set_title("Referencia de instancias")
axes[1].imshow(mascara_contacto, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Predicción pieza / fondo")
mostrar_cajas(axes[2], contacto["rgb"], cajas_contacto)
axes[2].set_title(f"Componentes: {len(cajas_contacto)}")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()
print("Piezas visibles:", len(cajas_de_etiquetas(contacto["instancias"])))
print("IoU de máscara:", round(iou_mascara(contacto["mascara"], mascara_contacto), 3))

**Discute:** ¿es correcto llamar a cada componente una instancia? ¿Qué aporta un modelo de segmentación de instancias que este procedimiento no aprende?

**Tu respuesta:** …

## Actividad 6 · Evaluar y documentar errores (15 min)

Con el modelo y umbral fijados evaluamos las seis escenas de prueba. Para cajas de una única clase usamos emparejamiento voraz uno a uno ordenado por IoU, con mínimo 0.5. Es una métrica docente de precisión/recall a un punto de operación; **no es AP/mAP ni reproduce todos los protocolos de evaluación**.

In [ ]:
def evaluar_cajas(reales, predichas, minimo_iou=0.5):
    candidatos = sorted([(iou_caja(a, b), i, j)
                          for i, a in enumerate(reales)
                          for j, b in enumerate(predichas)], reverse=True)
    usados_r, usados_p = set(), set()
    for valor, i, j in candidatos:
        if valor >= minimo_iou and i not in usados_r and j not in usados_p:
            usados_r.add(i)
            usados_p.add(j)
    tp = len(usados_r)
    return tp, len(predichas)-tp, len(reales)-tp

def resumen(escenas):
    total = np.zeros(3, dtype=int)
    ious = []
    for e in escenas:
        m, b = detectar(e)
        ious.append(iou_mascara(e["mascara"], m))
        total += evaluar_cajas(cajas_de_etiquetas(e["instancias"]), b)
    tp, fp, fn = total
    precision = tp / (tp+fp) if tp+fp else 0.0
    recall = tp / (tp+fn) if tp+fn else 0.0
    return {"IoU máscara media": round(float(np.mean(ious)), 3),
            "TP": int(tp), "FP": int(fp), "FN": int(fn),
            "Precisión cajas": round(float(precision), 3), "Recall cajas": round(float(recall), 3)}
print("Prueba independiente:", resumen(prueba))
print("Diagnóstico contacto:", resumen([contacto]))
poco_contraste = escena(500, bajo_contraste=True)
print("Diagnóstico color nuevo:", resumen([poco_contraste]))
# Los diagnósticos no se usan para elegir de nuevo el modelo o el umbral.
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, e, titulo in zip(axes, [prueba[0], contacto, poco_contraste],
                        ["Prueba", "Contacto", "Color nuevo"]):
    m, b = detectar(e)
    mostrar_cajas(ax, e["rgb"], b)
    ax.contour(m, levels=[0.5], colors=["yellow"], linewidths=0.7)
    ax.set_title(titulo)
plt.tight_layout()
plt.savefig("tema3_resultados.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: tema3_resultados.png")

**Entrega:** las tres escenas anotadas (cajas verdes y contornos amarillos), tabla de métricas y explicación de dos fallos. Si una predicción está vacía, no dibujará contorno ni caja: es un resultado que hay que analizar. Descarga `tema3_resultados.png` antes de cerrar Colab.

**Preguntas:** ¿una IoU semántica alta garantiza contar bien? ¿Qué ocurre al aumentar el mínimo de IoU para aceptar una caja? ¿Por qué un área de píxeles no es directamente mm²?

**Tu respuesta:** …

## Cierre

Distingue el umbral de puntuación por píxel, el filtro de área y el umbral de IoU para evaluar cajas. Son decisiones diferentes. Para un sistema real necesitaríamos fotografías representativas, anotaciones revisadas, clases y protocolo definidos, calibración geométrica si medimos y una evaluación independiente tras cerrar decisiones.